---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [13]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\Users\diana\Desktop\18.Ingineria_AI\echochamber-project-team3
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [14]:
import pandas as pd
import random

corpus = pd.read_json("C:\\Users\\diana\\Desktop\\18.Ingineria_AI\\echochamber-project-team3\\data\\cleaned\\corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[turcescu111] Da, noi suntem RESTUL lumii occidentale, adică grămăjoara aia care se duce, când
[digi24hd56] Mare dezamagire, domnilor! Ce cauta un scriitor, filozof sa faca politica?
[RecorderRomania] Păi de ce nu au sunat la poliție să reclame faptul ca se face o ilegalitate. Băi


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [15]:
# modifica dupa preferinte

AXA_1 = "anti_corruption"
AXA_2 = "people_vs_elite"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [16]:
AXA_1_DEFINITION = """
anti_corruption măsoară dacă textul exprimă o poziție clară împotriva corupției, 
cere pedepsirea politicienilor corupți, integritate în funcții publice sau o guvernare curată.
0 = absent (nu se menționează corupția sau furtul)
1 = prezent (se critică corupția sau hoția ca o problemă printre altele)
2 = dominant (întregul comentariu este axat pe denunțarea corupției și nevoia de curățenie morală)
""" 

AXA_2_DEFINITION = """
people_vs_elite măsoară dacă textul folosește dihotomia populistă dintre "oamenii simpli/popor" 
și "elitele corupte/cei de la putere", sugerând că cei din urmă ignoră complet nevoile cetățenilor.
0 = absent (nu se face o demarcație între popor și elite)
1 = prezent (se face referire tangențială la „cei de la putere” sau „oamenii de rând”)
2 = dominant (comentariul este construit complet pe conflictul dintre poporul asuprit și elita egoistă)
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [17]:
MINI_PROMPT = f"""
Ești un cercetător și analist politic expert, specializat în codarea riguroasă a discursului din social media. Răspunsul tău trebuie să fie strict obiectiv și să respecte în totalitate constrângerile de format solicitate.

SARCINĂ:
Adnotează următorul comentariu politic analizând exclusiv conținutul acestuia și contextul oferit, folosind două axe discursive specifice:
1. {AXA_1}
2. {AXA_2}

CÂMPURI:
- target = ținta politică principală sau entitatea vizată direct/implicit din comentariu.
- stance = poziția autorului față de target. Alege strict dintre variantele: pro / anti / neutru / ambiguu / none.
- tone = modul dominant de formulare a mesajului. Alege strict dintre variantele: acuzator / ironic / mobilizator / defensiv / afectiv / neutru.
- {AXA_1} = valoare numerică întreagă conform definiției (0 / 1 / 2).
- {AXA_2} = valoare numerică întreagă conform definiției (0 / 1 / 2).

DEFINIȚII AXE:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}

REGULI CRITICE DE CODARE:
1. Codează doar ce apare în mod direct sau implicit în comentariu, titlu sau canal. 
2. Nu inventa informații externe și nu presupune afilieri politice care nu reies din text.
3. Dacă nu există un target politic identificabil, folosește target="none" și stance="none".
4. Dacă textul este ironic sau sarcastic, codează sensul intenționat (ce vrea autorul de fapt să transmită), nu sensul literal al cuvintelor.
5. Pentru axele discursive respectă cu strictețe pragurile: 0 = absent, 1 = prezent, 2 = dominant.
6. Sub nicio formă nu încerca să atribui direct o bulă discursivă sau o tipologie globală (ex: suveranist, pro-european). Limitează-te doar la completarea axelor empirice.
7. Returnează textul EXCLUSIV sub formă de obiect JSON valid. Nu adăuga introduceri, explicații sau text liber înainte ori după blocul de cod.

FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""

print(MINI_PROMPT)


Ești un cercetător și analist politic expert, specializat în codarea riguroasă a discursului din social media. Răspunsul tău trebuie să fie strict obiectiv și să respecte în totalitate constrângerile de format solicitate.

SARCINĂ:
Adnotează următorul comentariu politic analizând exclusiv conținutul acestuia și contextul oferit, folosind două axe discursive specifice:
1. anti_corruption
2. people_vs_elite

CÂMPURI:
- target = ținta politică principală sau entitatea vizată direct/implicit din comentariu.
- stance = poziția autorului față de target. Alege strict dintre variantele: pro / anti / neutru / ambiguu / none.
- tone = modul dominant de formulare a mesajului. Alege strict dintre variantele: acuzator / ironic / mobilizator / defensiv / afectiv / neutru.
- anti_corruption = valoare numerică întreagă conform definiției (0 / 1 / 2).
- people_vs_elite = valoare numerică întreagă conform definiției (0 / 1 / 2).

DEFINIȚII AXE:

anti_corruption măsoară dacă textul exprimă o poziție cla

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [18]:
TESTS = corpus.sample(5, random_state=23)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
175,yt_KqrUotq1Obs_Ugy6tYmdfH0T2THArnB4AaABAg,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pace și prosperitate ( 28.10...,Bunul Dumnezeu să îl protejeze pe președintele...
387,yt_bee6nXyzJ_E_UgxVo3JOJXSmW55eVEx4AaABAg,@CălinGeorgescu-CanalulOficial,Călin Georgescu împreună cu Anca Alexandrescu ...,România este Gradina Maicii Domnului ! Tu doar...
63,yt_xSxPpRcYQp0_UgwqSeZe8Q_Q5g-cEJ94AaABAg,georgesimionoficial,Nimic fără Dumnezeu ! #cg1 #gs1 #impreuna #dem...,"Este pt prima data când voi vota pe cineva , p..."
315,yt_9rdHkT_RaIY_Ugza7-THxo7RRCpH6OB4AaABAg,CălinGeorgescu-CanalulOficial,Călin Georgescu - Impozit pe supraviețuire ( I...,VOI ȘTIȚI CE MAI PRODUCE ROMANIA IN 2025 ? Mă ...
231,yt_z5vgJ83XHjI_UgzjpuW8kv4vQ0f4nsR4AaABAg,spotmediaro,Nu m-am născut ministră. Vreau să fac ce am pr...,Doamnei ministru îi doresc multă forță să ducă...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [19]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [20]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [21]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Bunul Dumnezeu să îl protejeze pe președintele nostru Călin Georgescu ❤️❤️❤️

OUTPUT MODEL:
```json
{
  "target": "Călin Georgescu",
  "stance": "pro",
  "tone": "afectiv",
  "anti_corruption": 0,
  "people_vs_elite": 0
}
```
COMENTARIU:
România este Gradina Maicii Domnului ! Tu doar trebuie SĂ TACI SI SĂ FACI !!! Un DILIU care tot visează să aducă la conducere scursurile securiste, comuniste , tot felul de spioni ,,,,,doar vedeți că nu stă de vorbă cu nimeni ! Așa de ochii lumii cu câțiva zăpăciți care se țin după el ,,,,,,,,,,,,,, Maica Domnului a plâns când erau dărâmate bisericile ortodoxe din Romania BINE A FĂCUT Simion de a vândut alegerile ! Știa că nu o să aibă loc de prietenii lui Iliescu ,,,,,,,,

OUTPUT MODEL:
```json
{
  "target": "Călin Georgescu, Simion",
  "stance": "anti",
  "tone": "acuzator",
  "anti_corruption": 0,
  "people_vs_elite": 1
}
```
COMENTARIU:
Este pt prima data când voi vota pe cineva , pt a ajunge altcineva într-o funcție la conducerea țării

## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
- De ce le-ai ales?
- Modelul a returnat JSON corect?
- Care a fost cea mai mare problemă?
- Ce ai schimba în prompt?


Am ales axele "anti_corruption" și "people_vs_elite" deoarece sunt teme recutente in comentariile politice. Modelul a returnat un format JSON corect in toate cazurile, respectand astfel structura impusa. Cea mai mare problema ramane cea a incapabilitatii modelului de a intelege anumite nuante alte textelor si de a extrage informatii din comtext atunci cand nu sunt desrise in mod direct. Intr-o alta versiune de prompt as mari limitele people_vs_elite pentru a include si diferenta dintre noi si "strainii".